# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'external company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://sensemap.app")

Selecting relevant links for https://sensemap.app by calling gpt-5-nano


KeyError: 'links'

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Edge0/Edge0-35B-A3B-preview
Updated
about 18 hours ago
•
37.1k
•
3.3k
deepseek-ai/DeepSeek-V4.1-Flash
Updated
8 days ago
•
391k
•
3k
Qwen/Qwen3.8-27B
Updated
Aug 14
•
7.46M
•
15.5k
m-a-p/YuE2-3B
Updated
1 day ago
•
11.6k
•
716
TokenRhythm/NeoHorse-1-4B
Updated
8 days ago
•
19.8k
•
2.3

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("SenseMap", "https://sensemap.app/")

Selecting relevant links for https://sensemap.app/ by calling gpt-5-nano
Found 0 relevant links


'\nYou are looking at a company called: SenseMap\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nSenseMap — Find places that feel right for you\n\n\n## Relevant Links:\n'

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 22 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. It is a collaboration platform where the global machine learning (ML) community comes together to create, discover, and share models, datasets, and applications. With a vibrant ecosystem, Hugging Face hosts over **2 million models** and **500,000+ datasets**, supporting diverse AI and ML tasks openly and efficiently.

---

## What We Offer

- **Models:** Browse and collaborate on NLP, computer vision, audio, and multimodal AI models freely accessible on the platform.
- **Datasets:** Access and contribute to hundreds of thousands of curated and community-uploaded datasets to help train and evaluate models.
- **Spaces:** Deploy and share AI-powered web apps and demos hosted directly on Hugging Face to showcase innovative applications.
- **Buckets:** Efficient storage solutions for models, datasets, and other AI assets.
- **HuggingChat:** Advanced conversational AI tools integrated into the platform.
- **Enterprise Solutions:** Customized support including inference endpoints, storage buckets, and enterprise-grade collaboration tools.
- **Open Source:** Empowering developers and researchers with open source tools and frameworks accelerating AI innovation.

---

## Community & Collaboration

Hugging Face is more than a tech platform — it’s a community hub for AI research and engineering:

- Collaborative hosting of **unlimited public models and datasets**.
- Active involvement via forums, Discord, GitHub, and the official blog.
- Sharing daily research and papers to keep the community informed on latest advances.
- Engagement with ML tasks covering a wide variety of AI challenges.
- Events and resources to support newcomers and experts alike.

---

## Customers & Use Cases

- Researchers and educators accessing cutting-edge models and datasets.
- Enterprises implementing scalable AI solutions with professional support.
- Developers building and deploying AI applications quickly.
- AI startups leveraging shared community assets to accelerate innovation.
- Data scientists enhancing workflows with shared models and collaborative tools.

---

## Careers at Hugging Face

Join a passionate, forward-thinking team committed to growing the open AI ecosystem. Hugging Face hires talent dedicated to:

- Engineering open source AI tools and infrastructure.
- Developing new models and datasets to widen AI access.
- Supporting community growth and education.
- Enterprise partnership management and custom solutions.
- Product development and user experience innovation.

If you want to work at the forefront of AI technology with a collaborative and inclusive culture, Hugging Face offers exciting career opportunities in tech, research, and community engagement.

---

## Brand & Values

- **Inclusive Collaboration:** Building an open platform where anyone can contribute and benefit.
- **Innovation-Driven:** Constantly pushing the boundaries of AI development and deployment.
- **Transparency & Openness:** Empowering the community with open source and shared knowledge.
- **User-Centric:** Focusing on empowering individuals and organizations to leverage AI easily and effectively.

Official brand assets, including logos and color palettes, are available for partners and developers to use in their projects.

---

## Connect & Learn More

- **Explore AI Models & Apps:** https://huggingface.co/models  
- **Join the Community:** Discord, Forum, GitHub  
- **Enterprise Solutions:** Custom AI support and deployment  
- **Blog & Research:** Latest news, papers, and developments  
- **Careers:** Opportunities in AI, engineering, research, and more  

Hugging Face is where the future of AI is collaboratively built — come join the revolution!

---

*For more detailed information, visit* https://huggingface.co

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

## Bonus: what if the site is a Javascript-rendered app?

You may have noticed above that `select_relevant_links("https://sensemap.app")` and `get_brochure_user_prompt("SenseMap", "https://sensemap.app/")` came back with 0 relevant links and almost no landing page content. That's because sensemap.app is a React-style single page app: the raw HTML `requests` fetches is basically an empty shell, and the real content only appears after Javascript runs in a browser - the same problem flagged earlier with `openai.com`.

The fix: render the page with a real (headless) browser first. Two new functions were added to `scraper.py` for this - `fetch_rendered_website_contents` and `fetch_rendered_website_links` - using [Playwright](https://playwright.dev/python/).

One-time setup if you haven't already:
```
uv add playwright
uv run playwright install chromium
```

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import fetch_website_contents, fetch_rendered_website_contents, fetch_rendered_website_links

# Self-contained setup, in case earlier cells in this notebook weren't run in this kernel session
load_dotenv(override=True)
MODEL = 'gpt-5-nano'
openai = OpenAI()

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Compare: the plain scraper sees almost nothing, the rendered one sees the real page
print(fetch_website_contents("https://sensemap.app")[:200])
print("\n--- vs rendered ---\n")
print(fetch_rendered_website_contents("https://sensemap.app")[:500])
print(fetch_rendered_website_links("https://sensemap.app"))

SenseMap — Find places that feel right for you



--- vs rendered ---

SenseMap — Find places that feel right for you

SenseMap
Explore safely and simply
Log in
Sign up
Nature
Calm
✨ A calmer way to explore
The world is a lot.
We help you navigate it.
SenseMap shows you how loud, bright, and crowded any public space in Toronto is — before you arrive. Community reviews. AI insights. Built for people who need to know.
Explore the map
How it works
Comfortable
Moderate
Overwhelming
Search
Quiet spaces
Soft lighting
Low crowds
Outdoor areas
Before-noon spots
Built for
🧠
['https://reddit.com/r/autism', 'https://reddit.com/r/ADHD', 'https://forms.gle/F1NGPGdcKJR6Qg7u9']


In [2]:
# Same brochure pipeline as above, just swapping in the rendered fetchers

def select_relevant_links_rendered(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    links = fetch_rendered_website_links(url)
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    user_prompt += "\n".join(links)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    relevant_links = json.loads(result)
    print(f"Found {len(relevant_links['links'])} relevant links")
    return relevant_links


def fetch_page_and_all_relevant_links_rendered(url):
    contents = fetch_rendered_website_contents(url)
    relevant_links = select_relevant_links_rendered(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_rendered_website_contents(link["url"])
    return result


def get_brochure_user_prompt_rendered(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links_rendered(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt


def create_brochure_rendered(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt_rendered(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [3]:
create_brochure_rendered("SenseMap", "https://sensemap.app")

Selecting relevant links for https://sensemap.app by calling gpt-5-nano
Found 0 relevant links


# SenseMap Brochure

---

## Discover Places That Feel Right For You

**SenseMap** is an innovative sensory accessibility map designed to help neurodivergent individuals and sensory-sensitive people explore Toronto’s public spaces with confidence. By providing detailed information about how loud, bright, and crowded a location is before you arrive, SenseMap empowers users to choose spaces that match their comfort needs.

---

## Why SenseMap?

Navigating new places can be overwhelming, especially when sensory sensitivities play a significant role. SenseMap transforms exploration from a daunting gamble into a calm, informed experience.

- **Avoid surprises**: Check sensory scores like noise level, lighting, and crowd density before leaving your home.
- **Trusted insights**: Benefit from AI-generated analyses coupled with real community reviews.
- **Stay longer, enjoy more**: Feel confident arriving prepared, reducing anxiety and the likelihood of leaving early.

---

## Who Is SenseMap For?

SenseMap addresses the needs of various groups requiring sensory-aware planning:  
- **Autistic adults** — Plan outings tailored to your sensory preferences.  
- **Parents & Caregivers** — Scout and prepare visits for children or dependents.  
- **Support workers** — Match environments with client sensory needs.  
- **Sensory-sensitive individuals** — Anyone who benefits from knowing sensory details upfront.

---

## Core Features

- **Comprehensive sensory ratings:** Noise, crowd level, lighting, outdoor vs indoor, and time-of-day recommendations.  
- **Community-powered:** User-submitted reviews enhance accuracy and relevance.  
- **AI insights:** Smart technology helps analyze sensory qualities for quick understanding.  
- **Privacy-focused:** No ads, no tracking cookies, and masked session recordings ensure respectful data handling.

---

## SenseMap Culture

SenseMap is built with empathy and inclusivity at its core. The team prioritizes accessibility, community engagement, and privacy. By centering neurodivergent voices and sensory-sensitive needs, SenseMap fosters a welcoming space for innovation and collaboration.

---

## Careers and Joining SenseMap

SenseMap is growing and expanding beyond Toronto. They welcome individuals passionate about accessibility, AI technology, community impact, and inclusive design. Roles likely span software development, UX research, community outreach, and data science. If you want to help build tools that make the world more navigable and comfortable for all, SenseMap is an exciting place to grow your career.

---

## Plans and Expansion

Currently focused on Toronto, SenseMap is actively working to expand its map coverage to more cities, bringing its unique sensory accessibility approach to broader communities.

---

## Explore More

- **Try it yourself!** Explore the sensory map and plan your next outing with confidence.  
- **Join the community:** Submit reviews and share your experiences to improve the platform.  
- **Stay informed:** Visit the website for updates, terms of use, and privacy policies.

---

© 2026 SenseMap  
Privacy | Terms | Community Feedback  
*Making sensory-safe exploration simple, reliable, and accessible.*